In [1]:
!pip install langchain langchain-anthropic langchain-community \
    faiss-cpu sentence-transformers anthropic pandas openpyxl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [4]:
import os
import pandas as pd
from langchain_anthropic import ChatAnthropic
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

os.environ["ANTHROPIC_API_KEY"] = ""

llm = ChatAnthropic(model="claude-opus-4-5", max_tokens=800)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete.")

/tmp/ipykernel_1582/2350865761.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Setup complete.


In [6]:
FILE_PATH = "/content/swiftpay_fintech_dataset.xlsx"
xl = pd.ExcelFile(FILE_PATH)

documents = []

for sheet_name in xl.sheet_names:
    df = pd.read_excel(xl, sheet_name=sheet_name, header=2)
    for _, row in df.iterrows():
        label = str(row.iloc[0]) if pd.notna(row.iloc[0]) else ""
        if not label.strip() or label.strip().isupper():
            continue
        values = []
        for col, val in row.items():
            if pd.notna(val) and str(col) != row.index[0]:
                values.append(f"{col}: {val}")
        if values:
            text = f"Sheet: {sheet_name} | Metric: {label.strip()} | " + " | ".join(values[:14])
            documents.append(Document(
                page_content=text,
                metadata={"sheet": sheet_name, "metric": label.strip()}
            ))

print(f"Created {len(documents)} document chunks from {len(xl.sheet_names)} sheets")
print(f"Sample: {documents[0].page_content[:200]}")

Created 109 document chunks from 6 sheets
Sample: Sheet: Revenue Bridge | Metric: Payment Processing (85bps on TPV) | Jan-24: 15.7 | Feb-24: 16.3 | Mar-24: 15.1 | Apr-24: 17.9 | May-24: 19.4 | Jun-24: 21.1 | Jul-24: 22.5 | Aug-24: 16.8 | Sep-24: 23.1


In [7]:
# Split any long chunks further
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = splitter.split_documents(documents)

print(f"Chunks after splitting: {len(split_docs)}")

# Embed and store in FAISS
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("✅ Vector store built and retriever ready.")

Chunks after splitting: 109
✅ Vector store built and retriever ready.


In [8]:
# Prompt template — forces Claude to answer only from retrieved context
prompt = ChatPromptTemplate.from_template("""
You are a senior financial analyst for SwiftPay Corp.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have that data."
Be specific — cite actual numbers from the context.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# RAG chain: retrieve → format → prompt → LLM → parse
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready.")

✅ RAG chain ready.


In [9]:
def ask(question):
    print(f"❓ {question}")
    print("-" * 55)
    answer = rag_chain.invoke(question)
    print(answer)
    print("=" * 55 + "\n")

# Q1 — specific metric lookup
ask("What was SwiftPay's fraud rate trend in 2024?")

# Q2 — cross-sheet reasoning
ask("What were the top revenue streams and how did they perform?")

# Q3 — anomaly / risk question
ask("Were there any regulatory breaches or threshold violations?")

❓ What was SwiftPay's fraud rate trend in 2024?
-------------------------------------------------------
## SwiftPay's Fraud Rate Trend in 2024

SwiftPay's Overall Fraud Rate showed a **clear downward trend** throughout 2024, with one notable exception:

**Monthly Progression (in bps):**
- **Q1:** Started at 0.0018 (Jan), decreased to 0.0017 (Feb), slight uptick to 0.0019 (Mar)
- **Q2:** Continued declining from 0.0016 (Apr) to 0.0014 (Jun)
- **Q3:** Further improvement to 0.0013 (Jul), then **spiked to 0.0028 in August** (the year's peak), before dropping sharply to 0.0012 (Sep)
- **Q4:** Steady decline from 0.0011 (Oct) to 0.0009 (Dec) — the year's lowest point

**Key Observations:**
- **50% reduction** from January (0.0018) to December (0.0009)
- The **August anomaly** (0.0028) exceeded the threshold of 0.0025, but this was quickly corrected
- **FY Average:** 0.0015 bps — well below the 0.0025 threshold
- Excluding August, the trend was consistently downward throughout the year

The 